In [1]:
import os
import secrets
import requests
from urllib.parse import urlencode
from getpass import getpass

META_APP_ID = getpass("Enter Meta App ID: ")
META_APP_SECRET = getpass("Enter Meta App Secret: ")

# Must match the redirect URI configured in your Meta app.
REDIRECT_URI = input(
    "Enter configured redirect URI: "
).strip()

# Keep this value for the callback validation step.
OAUTH_STATE = secrets.token_urlsafe(32)

print("Credentials loaded.")
print("Redirect URI:", REDIRECT_URI)

Credentials loaded.
Redirect URI: https://localhost:8000/connections/instagram/callback/


In [2]:
INSTAGRAM_AUTH_URL = (
    "https://www.instagram.com/oauth/authorize"
)

params = {
    "client_id": META_APP_ID,
    "redirect_uri": REDIRECT_URI,
    "response_type": "code",
    "scope": "instagram_business_basic",
    "state": OAUTH_STATE,
}

authorization_url = (
    f"{INSTAGRAM_AUTH_URL}?{urlencode(params)}"
)

print("Open this URL in your browser:\n")
print(authorization_url)

Open this URL in your browser:

https://www.instagram.com/oauth/authorize?client_id=1625540195882562&redirect_uri=https%3A%2F%2Flocalhost%3A8000%2Fconnections%2Finstagram%2Fcallback%2F&response_type=code&scope=instagram_business_basic&state=O1y03DC-lWL4SJckXyjvDzrviL_Sv5p-g5mhZKDw5qw


In [ ]:
from urllib.parse import urlparse, parse_qs

callback_url = input(
    "Paste the complete redirected URL: "
).strip()

parsed_url = urlparse(callback_url)
callback_params = parse_qs(parsed_url.query)

returned_state = callback_params.get("state", [None])[0]
authorization_code = callback_params.get("code", [None])[0]
oauth_error = callback_params.get("error", [None])[0]

if oauth_error:
    raise RuntimeError(
        f"Instagram authorization failed: {oauth_error}"
    )

if returned_state != OAUTH_STATE:
    raise ValueError(
        "OAuth state mismatch. Restart the authorization flow."
    )

if not authorization_code:
    raise ValueError(
        "No authorization code found in the callback URL."
    )

print("OAuth state validated.")
print("Authorization code received.")

In [ ]:
TOKEN_URL = "https://api.instagram.com/oauth/access_token"

response = requests.post(
    TOKEN_URL,
    data={
        "client_id": META_APP_ID,
        "client_secret": META_APP_SECRET,
        "grant_type": "authorization_code",
        "redirect_uri": REDIRECT_URI,
        "code": authorization_code,
    },
    timeout=30,
)

print("HTTP status:", response.status_code)

if response.ok:
    token_data = response.json()

    access_token = token_data.get("access_token")

    print("Token exchange successful.")
    print("Instagram user ID:", token_data.get("user_id"))
    print("Token type:", token_data.get("token_type"))
    print("Expires in:", token_data.get("expires_in"), "seconds")
    print("Access token received:", bool(access_token))
else:
    print("Token exchange failed.")
    print(response.text)

In [ ]:
if not access_token:
    raise RuntimeError(
        "No access token available. Complete Step 5 successfully."
    )

ACCOUNT_URL = "https://graph.instagram.com/v23.0/me"

response = requests.get(
    ACCOUNT_URL,
    params={
        "fields": "user_id,username",
        "access_token": access_token,
    },
    timeout=30,
)

print("HTTP status:", response.status_code)

if response.ok:
    account_data = response.json()

    print("Instagram account connected successfully!")
    print("Account information:", account_data)
else:
    print("Account lookup failed.")
    print(response.text)

In [ ]:
if not access_token:
    raise RuntimeError("Complete the token exchange first.")

LONG_LIVED_TOKEN_URL = (
    "https://graph.instagram.com/access_token"
)

response = requests.get(
    LONG_LIVED_TOKEN_URL,
    params={
        "grant_type": "ig_exchange_token",
        "client_secret": META_APP_SECRET,
        "access_token": access_token,
    },
    timeout=30,
)

print("HTTP status:", response.status_code)

if response.ok:
    long_lived_data = response.json()

    long_lived_token = long_lived_data.get("access_token")

    print("Long-lived token exchange successful.")
    print("Expires in:", long_lived_data.get("expires_in"), "seconds")
else:
    print("Long-lived token exchange failed.")
    print(response.text)